In [1]:
import pandas as pd
import os

files_map = {
    "Madinah": "Madinah_Region_temp.csv",
    "Makkah": "Makkah_Region_temp.csv",
    "Najran": "Najran_Region_temp.csv",
    "Northern Borders": "Northern_Borders_Region_temp.csv",
    "Qassim": "Qassim_Region_temp.csv",
    "Riyadh": "Riyadh_Region_temp.csv",
    "Tabuk": "Tabuk_Region_temp.csv",
    "Al Baha": "AlBaha_Region_temp.csv",
    "Al Jouf": "AlJouf_Region_temp.csv",
    "Asir": "Asir_Region_temp.csv",
    "Eastern": "Eastern_Region_temp.csv",
    "Hail": "Hail_Region_temp.csv",
    "Jazan": "Jazan_Region_temp.csv"
    }

all_regions_data = []

print("processing...")

for region_name, file_name in files_map.items():
    if os.path.exists(file_name):
        print(f"Processing {region_name} Region from {file_name}...")

        df = pd.read_csv(file_name, skiprows=3)

        df['time'] = pd.to_datetime(df['time'])

        monthly_avg = df.resample('ME', on='time').mean(numeric_only=True)

        monthly_data = monthly_avg.reset_index()
        monthly_data['Region'] = region_name
        monthly_data['Month'] = monthly_data['time'].dt.month
        monthly_data['Year'] = monthly_data['time'].dt.year

        temp_col = [c for c in monthly_data.columns if 'temperature' in c.lower()][0]

        final_df = monthly_data[['Region', 'Year', 'Month', temp_col]].copy()
        final_df.rename(columns={temp_col: 'Avg_Temperature'}, inplace=True)

        final_df['Avg_Temperature'] = final_df['Avg_Temperature'].round(1)

        all_regions_data.append(final_df)

    else:
        print(f"Warning: File {file_name} not found. Skipped.")

if all_regions_data:
    unified_dataset = pd.concat(all_regions_data, ignore_index=True)

    output_filename = 'Saudi_Regions_Monthly_Temp.csv'
    unified_dataset.to_csv(output_filename, index=False)

    print("\nSuccess. Saved to", output_filename)
else:
    print("\nNo files were processed.")

processing...
Processing Madinah Region from Madinah_Region_temp.csv...
Processing Makkah Region from Makkah_Region_temp.csv...
Processing Najran Region from Najran_Region_temp.csv...
Processing Northern Borders Region from Northern_Borders_Region_temp.csv...
Processing Qassim Region from Qassim_Region_temp.csv...
Processing Riyadh Region from Riyadh_Region_temp.csv...
Processing Tabuk Region from Tabuk_Region_temp.csv...
Processing Al Baha Region from AlBaha_Region_temp.csv...
Processing Al Jouf Region from AlJouf_Region_temp.csv...
Processing Asir Region from Asir_Region_temp.csv...
Processing Eastern Region from Eastern_Region_temp.csv...
Processing Hail Region from Hail_Region_temp.csv...
Processing Jazan Region from Jazan_Region_temp.csv...

Success. Saved to Saudi_Regions_Monthly_Temp.csv


In [2]:
import pandas as pd

region_mapping = {
    'Riyadh': 'Riyadh', 'Makkah': 'Makkah', 'Madinah': 'Madinah',
    'Qassim': 'Qassim', 'Eastern': 'Eastern Region', 'Asir': 'Aseer',
    'Tabuk': 'Tabuk', 'Hail': 'Hail', 'Northern Borders': 'Northern Borders',
    'Jazan': 'Jazan', 'Najran': 'Najran', 'Al Baha': 'Al-Baha', 'Al Jouf': 'Al-Jouf'
}

temp_df = pd.read_csv('Saudi_Regions_Monthly_Temp.csv')
winter_months = [11, 12, 1, 2]

temp_summary = {}
for temp_reg, stat_reg in region_mapping.items():
    region_data = temp_df[temp_df['Region'] == temp_reg]
    winter_data = region_data[region_data['Month'].isin(winter_months)]
    rest_data = region_data[~region_data['Month'].isin(winter_months)]

    temp_summary[stat_reg] = {
        'Winter': round(winter_data['Avg_Temperature'].mean(), 1),
        'Rest_of_Year': round(rest_data['Avg_Temperature'].mean(), 1)
    }

excel_file = 'Household_Energy_Statistics_2024.xlsx'

def read_sheet(sheet_name, cols):
    df = pd.read_excel(excel_file, sheet_name=sheet_name, skiprows=4)
    df = df.dropna(subset=[df.columns[1]]).iloc[:, cols]
    df.iloc[:, 0] = df.iloc[:, 0].astype(str).str.strip()
    for i in range(1, len(cols)):
        df.iloc[:, i] = pd.to_numeric(df.iloc[:, i], errors='coerce').fillna(0)
    return df

# Read appliance consumption in hours
ac_df = read_sheet('1-16', [1, 4, 5])
ac_df.columns = ['Region', 'AC_Winter', 'AC_Rest']

heater_df = read_sheet('1-14', [1, 2, 3])
heater_df.columns = ['Region', 'Heater_Winter', 'Heater_Rest']

water_heater_df = read_sheet('1-15', [1, 2, 3])
water_heater_df.columns = ['Region', 'WH_Winter', 'WH_Rest']

# Read thermal insulation
insulation_df = read_sheet('1-20', [1, 2])
insulation_df.columns = ['Region', 'Insulation_Yes_Pct']
insulation_df['Insulation_Yes_Pct'] = (insulation_df['Insulation_Yes_Pct'] * 100)

# Read cooking sources divided by dwelling type
dwelling_sheets = {
    'Villa': '2-9',
    'Floor in Villa': '2-10',
    'Traditional House': '2-11',
    'Floor in Traditional': '2-12',
    'Apartment': '2-13'
}

cooking_data = []
for dwelling_type, sheet in dwelling_sheets.items():
    df = read_sheet(sheet, [1, 2, 3])
    df.columns = ['Region', 'Cooking_Gas_Pct', 'Cooking_Elec_Pct']
    df['Dwelling_Type'] = dwelling_type
    cooking_data.append(df)

cooking_df = pd.concat(cooking_data, ignore_index=True)
cooking_df['Cooking_Gas_Pct'] = (cooking_df['Cooking_Gas_Pct'] * 100)
cooking_df['Cooking_Elec_Pct'] = (cooking_df['Cooking_Elec_Pct'] * 100)

# Build the dataset (Region x Season x Dwelling Type)
csv_rows = []

for reg in region_mapping.values():
    if reg not in temp_summary: continue

    ac_row = ac_df[ac_df['Region'] == reg].iloc[0]
    heat_row = heater_df[heater_df['Region'] == reg].iloc[0]
    wh_row = water_heater_df[water_heater_df['Region'] == reg].iloc[0]
    insul_row = insulation_df[insulation_df['Region'] == reg].iloc[0]

    for season in ['Winter', 'Rest_of_Year']:
        for dwelling in dwelling_sheets.keys():
            cook_row = cooking_df[(cooking_df['Region'] == reg) & (cooking_df['Dwelling_Type'] == dwelling)].iloc[0]

            csv_rows.append({
                'Region': reg,
                'Season': season,
                'Dwelling_Type': dwelling,
                'Average_Temp_C': round(temp_summary[reg][season], 1),
                'Split_AC_hours_per_week': round(ac_row['AC_Winter' if season == 'Winter' else 'AC_Rest'], 1),
                'Radiant_Heater_hours_per_week': round(heat_row['Heater_Winter' if season == 'Winter' else 'Heater_Rest'], 1),
                'Water_Heater_hours_per_week': round(wh_row['WH_Winter' if season == 'Winter' else 'WH_Rest'], 1),
                'Insulation_Yes_Pct': round(insul_row['Insulation_Yes_Pct'], 1),
                'Cooking_Gas_Pct': round(cook_row['Cooking_Gas_Pct'], 1),
                'Cooking_Elec_Pct': round(cook_row['Cooking_Elec_Pct'], 1)
            })

final_df = pd.DataFrame(csv_rows)

final_df = final_df.round(1)

final_df.to_csv('household_consumption_baseline.csv', index=False, encoding='utf-8-sig')

print("Success. Saved to household_consumption_baseline.csv")

Success. Saved to household_consumption_baseline.csv
